# 01 — Tokenizers, AutoModel, generate(), and a LoRA Concept Check

**Overview:** load a tiny pretrained tokenizer + causal LM, tokenize text and inspect token
IDs/attention masks, run `generate()`, inspect model configuration, preprocess a Hugging Face
`Dataset` with the tokenizer, and compare LoRA's parameter count to full fine-tuning
conceptually.

**Required libraries:** `transformers`, `datasets`, `torch`.

```
!pip install -q transformers
```

**Note:** this notebook uses `sshleifer/tiny-gpt2` — a genuinely tiny (102K parameter, ~500KB)
GPT-2-architecture model with *randomly initialized* weights, used throughout the Hugging Face
ecosystem for fast testing. Its generated text will be nonsensical (the point here is the
*mechanics* — tokenization, generation loop, config — not output quality). Swap in a real small
model (e.g. `distilgpt2`, ~350MB) for coherent output if you have the bandwidth.

Connects to: `17_deep_learning_architectures` (causal masking, the Transformer this model IS),
`21_pytorch_data` (attention masks for padding), and Day 1's `04_datasets` (the `Dataset` being
tokenized here). This closes Day 3 and the full curriculum, handing off directly to this
workspace's `llm-fine-tuning/` curriculum.

### Abbreviations used in this notebook

| Term | Full form | What it is |
|---|---|---|
| **LoRA** | Low-Rank Adaptation | Fine-tunes a model by training small low-rank matrices instead of all weights. |
| **LM** | Language Model | Model that assigns probabilities to token sequences / predicts the next token. |
| **ID** | Identifier | Integer index of a token (token ID) or record. |
| **GPT** | Generative Pre-trained Transformer | Decoder-only Transformer family trained to predict the next token. |
| **AI** | Artificial Intelligence | Systems that perform tasks normally needing human intelligence; ML is a subset. |
| **ML** | Machine Learning | Learning patterns from data to make predictions, instead of hand-coding rules. |
| **QLoRA** | Quantized LoRA | LoRA on top of a 4-bit quantised base model to save memory. |
| **LLM** | Large Language Model | Very large Transformer trained on text to predict the next token (e.g. GPT). |
| **RAG** | Retrieval-Augmented Generation | Fetch relevant documents and feed them to an LLM so answers are grounded in them. |

In [ ]:
%pip install -q transformers datasets

## 0. Environment

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers
print("transformers:", transformers.__version__)

MODEL_NAME = "sshleifer/tiny-gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    # GPT-2's tokenizer has no dedicated pad token by convention -- reuse eos_token, the
    # standard practice for GPT-2-family models (padding never attends to real content anyway,
    # since the attention mask marks it out).
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()   # recap from 20_pytorch_models: always eval() mode for inference

print("vocab size:", tokenizer.vocab_size)
print("total model parameters:", sum(p.numel() for p in model.parameters()))


transformers: 4.57.2


W1004 14:30:59.975000 91325 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


W1004 14:31:00.007000 91325 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W1004 14:31:00.043000 91325 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


vocab size: 50257
total model parameters: 102714


## 1. Tokenization: text -> token IDs

💡 **Intuition:** each token ID is a row index into the model's embedding table
(`nn.Embedding`, from `20_pytorch_models`) -- the tokenizer and model MUST agree on what each
ID means, which is why they're always loaded as a matched pair.

In [2]:
text = "Deep learning models process tokens, not raw text."
encoded = tokenizer(text, return_tensors="pt")

print("input_ids:", encoded["input_ids"])
print("attention_mask:", encoded["attention_mask"])

tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
print("\ntoken-by-token breakdown:")
for token_id, token_str in zip(encoded["input_ids"][0].tolist(), tokens):
    print(f"  id={token_id:6d}  token={token_str!r}")

assert encoded["input_ids"].shape[1] == len(tokens)


input_ids: tensor([[29744,  4673,  4981,  1429, 16326,    11,   407,  8246,  2420,    13]])
attention_mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

token-by-token breakdown:
  id= 29744  token='Deep'
  id=  4673  token='Ġlearning'
  id=  4981  token='Ġmodels'
  id=  1429  token='Ġprocess'
  id= 16326  token='Ġtokens'
  id=    11  token=','
  id=   407  token='Ġnot'
  id=  8246  token='Ġraw'
  id=  2420  token='Ġtext'
  id=    13  token='.'


## 2. Attention masks for padding (recap from `21_pytorch_data`)

⚠️ **Common Mistake:** forgetting the attention mask when batching variable-length sequences
-- padded positions would otherwise leak into the attention computation from
`17_deep_learning_architectures`.

In [3]:
sentences = ["Short one.", "This is a considerably longer sentence than the first one."]
batch = tokenizer(sentences, return_tensors="pt", padding=True)

print("input_ids:\n", batch["input_ids"])
print("\nattention_mask (0 = padding, 1 = real token):\n", batch["attention_mask"])

# The shorter sentence should have trailing padding, marked 0 in the attention mask.
shorter_sentence_mask = batch["attention_mask"][0]
assert shorter_sentence_mask.sum() < shorter_sentence_mask.numel()   # some padding exists
print(f"\nshorter sentence: {shorter_sentence_mask.sum().item()} real tokens out of "
      f"{shorter_sentence_mask.numel()} padded positions")


input_ids:
 tensor([[16438,   530,    13, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256],
        [ 1212,   318,   257, 15394,  2392,  6827,   621,   262,   717,   530,
            13]])

attention_mask (0 = padding, 1 = real token):
 tensor([[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

shorter sentence: 3 real tokens out of 11 padded positions


## 3. `generate()`: the autoregressive loop, wrapped

🎯 **Interview Tip:** `generate()` is repeatedly predicting the next token and feeding it back
in -- exactly the causal/autoregressive decoding concept from `17_deep_learning_architectures`,
just implemented for you.

In [4]:
prompt = "The history of artificial intelligence"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():   # recap from 19_pytorch_autograd: no gradients needed for inference
    output_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,          # greedy decoding -- deterministic, simplest strategy
        pad_token_id=tokenizer.eos_token_id,
    )

generated_text = tokenizer.decode(output_ids[0])
print("prompt:", prompt)
print("generated (random-weight model -> expect gibberish, not coherent text):")
print(generated_text)

assert output_ids.shape[1] > inputs["input_ids"].shape[1]   # new tokens were actually generated


prompt: The history of artificial intelligence
generated (random-weight model -> expect gibberish, not coherent text):
The history of artificial intelligence stairs stairs stairs stairs stairs stairs stairs stairs stairs stairs


## 4. Model configuration

💡 **Key Idea:** every architectural hyperparameter from `17_deep_learning_architectures`
(layers, hidden size, attention heads) is stored in the model's config, loaded alongside the
weights.

In [5]:
config = model.config
print("number of layers:", config.n_layer)
print("hidden size:", config.n_embd)
print("number of attention heads:", config.n_head)
print("vocab size:", config.vocab_size)

assert config.n_embd % config.n_head == 0   # hidden size must divide evenly across heads
print(f"\neach attention head handles {config.n_embd // config.n_head} dimensions "
      f"(hidden_size / num_heads) -- exactly the multi-head attention split from "
      f"17_deep_learning_architectures.")


number of layers: 2
hidden size: 2
number of attention heads: 2
vocab size: 50257

each attention head handles 1 dimensions (hidden_size / num_heads) -- exactly the multi-head attention split from 17_deep_learning_architectures.


## 5. Preprocessing a Hugging Face `Dataset` with this tokenizer (recap from Day 1's `04_datasets`)

In [6]:
from datasets import Dataset

raw_texts = Dataset.from_dict({"text": [
    "Tokenizers turn text into numbers.",
    "Attention masks handle padding.",
    "Generate implements autoregressive decoding.",
]})

def tokenize_batch(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=16)

tokenized = raw_texts.map(tokenize_batch, batched=True)
print(tokenized)
print("\nfirst example's input_ids:", tokenized[0]["input_ids"])

assert "input_ids" in tokenized.column_names
assert "attention_mask" in tokenized.column_names
assert len(tokenized[0]["input_ids"]) == 16   # padded/truncated to max_length


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 3
})

first example's input_ids: [30642, 11341, 1210, 2420, 656, 3146, 13, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256]


## 6. LoRA, conceptually: how few parameters does fine-tuning actually need?

💡 **Intuition:** LoRA freezes the full model and adds small low-rank matrices
(`A: d x r`, `B: r x d`, for a weight matrix of shape `d x d`) to approximate the WEIGHT UPDATE
needed for a new task -- if `r << d`, this is a tiny fraction of the original parameter count.
This is a parameter-count illustration only; actually attaching trainable LoRA adapters is
`llm-fine-tuning/02_lora_qlora_peft`'s hands-on subject.

In [7]:
total_model_params = sum(p.numel() for p in model.parameters())

# Illustrative: if we LoRA-adapted every linear layer's weight matrices with a small rank...
def lora_param_count(model, rank):
    count = 0
    for module in model.modules():
        if isinstance(module, torch.nn.Linear):
            d_out, d_in = module.weight.shape
            count += rank * d_in + rank * d_out   # A: (rank, d_in), B: (d_out, rank)
    return count

for rank in [1, 4, 8]:
    lora_params = lora_param_count(model, rank)
    fraction = lora_params / total_model_params
    print(f"rank={rank}: {lora_params:,} LoRA params == {fraction:.1%} of the model's "
          f"{total_model_params:,} total parameters")

assert lora_param_count(model, 1) < total_model_params
print("\nOn a real large language model (billions of parameters, not this tiny test model), "
      "this fraction becomes dramatically smaller -- often well under 1% -- which is exactly "
      "why LoRA makes fine-tuning large models tractable on modest hardware.")


rank=1: 50,259 LoRA params == 48.9% of the model's 102,714 total parameters
rank=4: 201,036 LoRA params == 195.7% of the model's 102,714 total parameters
rank=8: 402,072 LoRA params == 391.4% of the model's 102,714 total parameters

On a real large language model (billions of parameters, not this tiny test model), this fraction becomes dramatically smaller -- often well under 1% -- which is exactly why LoRA makes fine-tuning large models tractable on modest hardware.


## 7. Connection to what's next

This closes Day 3 and the full "AI/ML Engineer Fundamentals" curriculum. Every concept here —
tokenization, attention masks, causal generation, model config, LoRA's parameter-efficiency
idea — is picked up and gone deep on on in this workspace's `llm-fine-tuning/` curriculum:
`01_sft_fundamentals` (supervised fine-tuning), `02_lora_qlora_peft` (hands-on LoRA/QLoRA),
`03_continued_pretraining`, `04_preference_optimization_rl`, `05_distillation_compression`, and
`06_scaling_eval_production`.

## 8. Interview questions
1. Why must a tokenizer and a model always come from the same pretrained checkpoint?
2. What does the attention mask prevent during a forward pass on a padded batch?
3. Walk through what `generate()` is doing, step by step, in terms of `17_deep_learning_architectures`'s causal masking.
4. Why does LoRA's parameter count scale with `rank * (d_in + d_out)` rather than `d_in * d_out`?
5. Why would a production system validate a model's config at startup rather than assuming it matches expectations?

## 9. Exercises
1. Tokenize a batch of 5 sentences of very different lengths and verify every attention mask correctly identifies padding.
2. Generate text with `do_sample=True` instead of greedy decoding and compare outputs across multiple runs with different seeds.
3. Compute the LoRA parameter fraction for a hypothetical model with `n_embd=4096` (a realistic large-LLM hidden size) at rank 8, and compare to this notebook's tiny-model numbers.

## 10. Summary

✅ `AutoTokenizer`/`AutoModelForCausalLM` give a consistent interface across any pretrained
model; attention masks handle padding correctly; `generate()` automates the autoregressive
decoding loop; LoRA's low-rank adapters make fine-tuning large models parameter-efficient.
This closes the AI/ML Engineer Fundamentals curriculum (Days 1-3) — continue to this
workspace's `llm-fine-tuning/` curriculum for LLMs, fine-tuning, RAG, and inference engineering.